In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE F32ListAdd/Copy BITEXACT TEST (PURE TKV) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + 2 .tkv C .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
for nm in ['f32_list_add_cpu','f32_list_copy_cpu']:
    cc=ex+'/'+nm+'.cpu.c'; so=ex+'/'+nm+'.so'
    txt=open(cc).read()
    print(nm,'C INJECTED =', 'tkv_'+nm in txt, '(expect False)')
    p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
    print(nm,'build rc=',p2.returncode)
    if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare F32ListAdd / F32ListCopy
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); II=ctypes.c_int
args=[PP,II,PP,II,II]
libn=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so')
for kn in ['F32ListAdd','F32ListCopy']:
    native=getattr(libn,kn); native.argtypes=args; native.restype=II
    ex='/content/TVcpu/Release/cuda_backend/examples'
    so='f32_list_add_cpu.so' if kn=='F32ListAdd' else 'f32_list_copy_cpu.so'
    tkv=ctypes.CDLL(ex+'/'+so).F32ListAdd if kn=='F32ListAdd' else ctypes.CDLL(ex+'/'+so).F32ListCopy
    tkv.argtypes=args; tkv.restype=II
    rng=np.random.default_rng(23)
    for n in [17,64,1000,4097]:
        src=rng.standard_normal(n).astype(np.float32)
        d0=rng.standard_normal(n).astype(np.float32)
        a=d0.copy(); c=d0.copy()
        if kn=='F32ListCopy':
            a[:]=0; c[:]=0
        native(a.ctypes.data_as(PP),a.size,src.ctypes.data_as(PP),src.size,n)
        tkv  (c.ctypes.data_as(PP),c.size,src.ctypes.data_as(PP),src.size,n)
        d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
        print('%s n=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(kn,n,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> F32ListAdd/Copy pure .tkv reproduces native (NO C injection).')